Carregando os dados e separando df's de treino e de teste.

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dengue_pipeline").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from dengue_pipeline import DengueDataCleaner
from dengue_pipeline.models import GradientBoostingDiseaseClassifier, MLPDiseaseClassifier
from dengue_pipeline.paths import MODELS_DIR, MODEL_FIGURES_DIR, ML_PREPROCESS_PATH

from sklearn.metrics import classification_report, confusion_matrix

MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def salvar_modelo(modelo, nome_arquivo):
    caminho = MODELS_DIR / nome_arquivo
    joblib.dump(modelo, caminho)
    print(f"Modelo salvo em: {caminho}")
    return caminho

def plotar_importancias(importancias, titulo, nome_arquivo, top_n=30):
    dados = importancias.sort_values(ascending=False).head(top_n).sort_values()
    fig, ax = plt.subplots(figsize=(11, 8))
    ax.barh(dados.index, dados.values, color="#2563eb", alpha=0.9)
    ax.set_title(titulo, fontsize=14, fontweight="bold", pad=12)
    ax.set_xlabel("Importância")
    ax.set_ylabel("")
    ax.grid(axis="x", linestyle="--", alpha=0.22)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.tight_layout()
    caminho = MODEL_FIGURES_DIR / nome_arquivo
    fig.savefig(caminho, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Gráfico salvo em: {caminho}")

cleaner = DengueDataCleaner()
df = cleaner.transformar_ml()

# Salva os encoders ajustados (ocupação e UF) + a mediana de days_to_notification
# para a API reproduzir o mesmo pré-processamento na hora da predição.
joblib.dump(
    {
        "occupation_encoder": cleaner.occupation_encoder,
        "residence_state_encoder": cleaner.residence_state_encoder,
        "days_to_notification_median": cleaner.days_to_notification_median,
    },
    ML_PREPROCESS_PATH,
)
print(f"Pré-processamento salvo em: {ML_PREPROCESS_PATH}")

df_train = df[(df["notification_year"].isin([2017, 2018])) | ((df["notification_year"] == 2019) & (df["notification_month"] <= 5))].copy()
df_test = df[(df["notification_year"] == 2019) & (df["notification_month"] >= 6)].copy()

print("Treino:", df_train.shape)
print("Teste:", df_test.shape)

print("Proporção treino:", (len(df_train) / len(df))*100)
print("Proporção teste:", (len(df_test) / len(df))*100)

df.columns

Definindo o target e eliminando colunas que não são apropriadas para o modelo

In [ ]:
target = "final_classification"

y_train = df_train[target]
y_test = df_test[target]

cols_remove = ['final_classification']

X_train = df_train.drop(columns = cols_remove)
X_test = df_test.drop(columns=cols_remove)

X_train = X_train.select_dtypes(include=["number"])
X_test = X_test[X_train.columns]

X_train = X_train.astype("float32")
X_test = X_test.astype("float32")

In [ ]:
administrative_features = [
    "notification_year",
    "notification_month",
    "symptom_epi_year",
    "notif_municipality",
    "notif_health_region",
    "health_facility",
]

X_train = X_train.drop(columns=administrative_features, errors="ignore")
X_test = X_test.drop(columns=administrative_features, errors="ignore")

Implementação da MLP com embeddings

In [ ]:
# A arquitetura interna é a mesma do projeto original: embeddings para
# categóricas, BatchNorm e camadas 1024 -> 512 -> 256 -> 128.
# O wrapper atual mantém os dados na CPU e envia apenas cada batch para a GPU.
modelo_mlp = MLPDiseaseClassifier(
    hidden_layers=(1024, 512, 256, 128),
    embedding_dropout=0.1,
    hidden_dropout=0.2,
    batch_size=16384,
    learning_rate=1e-3,
    weight_decay=1e-4,
    max_epochs=150,
    patience=10,
    validation_size=0.1,
    device="auto",
    random_state=42,
)

modelo_mlp.fit(X_train, y_train)
y_pred_mlp = modelo_mlp.predict(X_test)

print(classification_report(y_test, y_pred_mlp))
print(confusion_matrix(y_test, y_pred_mlp))

resultados_limiares_mlp = modelo_mlp.evaluate(
    X_test,
    y_test,
    thresholds=[0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80],
)
display(resultados_limiares_mlp)

salvar_modelo(modelo_mlp, "mlp.joblib")
importancias_mlp = modelo_mlp.permutation_feature_importance(
    X_test,
    y_test,
    sample_size=2_000,
    n_repeats=3,
)
plotar_importancias(
    importancias_mlp,
    "MLP - Top 30 Features por Permutação",
    "mlp_feature_importance.png",
)

Implementação de XGBoost

In [ ]:
# Teste da classe adaptada usando XGBoost.
# O XGBoost trata valores ausentes nativamente.
modelo_xgb = GradientBoostingDiseaseClassifier(
    model="xgb",
    fast_train=False,
    device="cuda"
)

modelo_xgb.fit(
    X_train,
    y_train,
    n_trials=200,
    tuning_sample_size=200_000,
)
y_pred_xgb = modelo_xgb.predict(X_test)

print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))

resultados_limiares_xgb = modelo_xgb.evaluate(
    X_test,
    y_test,
    thresholds=[0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80],
)
display(resultados_limiares_xgb)

salvar_modelo(modelo_xgb, "xgboost.joblib")
importancias_xgb = modelo_xgb.feature_importance(importance_type="gain")
plotar_importancias(
    importancias_xgb,
    "XGBoost - Top 30 Features por Ganho",
    "xgboost_feature_importance.png",
)

In [ ]:
# Teste da classe adaptada usando LightGBM.
# O LightGBM trata valores ausentes nativamente.
modelo_lgb = GradientBoostingDiseaseClassifier(
    model="lgbm",
    fast_train=False,
    device="cuda"
)

modelo_lgb.fit(
    X_train,
    y_train,
    n_trials=200,
    tuning_sample_size=200_000,
)
y_pred_lgb = modelo_lgb.predict(X_test)

print(classification_report(y_test, y_pred_lgb))
print(confusion_matrix(y_test, y_pred_lgb))

resultados_limiares_lgb = modelo_lgb.evaluate(
    X_test,
    y_test,
    thresholds=[0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80],
)
display(resultados_limiares_lgb)

salvar_modelo(modelo_lgb, "lightgbm.joblib")
importancias_lgb = modelo_lgb.feature_importance(importance_type="gain")
plotar_importancias(
    importancias_lgb,
    "LightGBM - Top 30 Features por Ganho",
    "lightgbm_feature_importance.png",
)